# SPW-inspired code experiment — frozen StarCoder2-3B
**Approximation, not published SPW-MIA.** The complete method equations, algorithms, default shortening criteria and hyperparameters could not be retrieved from DOI [10.1016/j.knosys.2026.116100](https://doi.org/10.1016/j.knosys.2026.116100). Only the publisher preview was accessible. `research_access.json` records the missing material and retrieval attempts. No equation below is attributed to a paper equation number. This notebook implements the user-authorized, explicitly named fallback.

Our single frozen procedure uses dynamic halving until the greedy prediction changes; compression/probability-stability token weights; and a negative weighted Bernoulli Jensen–Shannon divergence between true-token and greedy-token probabilities. Higher scores are member-like by design, fixed before labels. See packaged README for every equation. No reference model, training, alternative score or hyperparameter grid.

Both modes validate complete saved SERSEM manifests. Analyze the first **256 eligible target tokens**, with seed **42**, uniformly across Go, Java, Python, Ruby and Rust. This is a compute-budget adaptation; the original paper's context length was unavailable. All target inference runs manually on Kaggle.

In [ ]:
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
UPLOAD_SOURCE = INPUT_ROOT / 'EDIT-spw-source-dataset'
UPLOAD_ZIP = UPLOAD_SOURCE / 'spw_kaggle_source.zip'
PROJECT_ROOT = WORK_ROOT / 'spw-inspired-code-source-v1'
BASELINE_RUN = ''  # discover exactly one complete saved baseline run; set if ambiguous
PATH_OVERRIDES = dict(shadow='', validation='', calibration='', heldout='', scores='', config='')
ADAPTIVE_SCORES = ''  # intended heldout_adaptive_scores.parquet; full saved scores recommended
MODE = 'smoke'  # 'smoke' or 'full'
SMOKE_SIZES = dict(shadow=80, validation=20, calibration=20, heldout=30)
RUN_ID = f'spw-inspired-code-{MODE}-v1'
OUT = WORK_ROOT / 'spw_outputs' / RUN_ID
RESUME_FROM = ''  # attach saved OUT from this SAME scientific configuration/mode
CACHE_REUSE_FROM = ''  # optional saved spw_cache.sqlite, e.g. smoke -> fresh full run
MODEL_CACHE_DIR = ''  # empty inherits baseline HF model cache root
MODEL_LOCAL_FILES_ONLY = True
ALLOW_MODEL_DOWNLOAD = False  # True explicitly permits pinned target download on Kaggle
DATASET_CACHE_DIR = WORK_ROOT / 'hf_cache/datasets'
SOURCE_ACCESS = 'streaming'  # streaming | cached (requires existing Arrow cache)
SESSION_BUDGET_MINUTES = 480
DISK_RESERVE_GIB = 3.
WORK_ROOT.mkdir(parents=True,exist_ok=True)
# Scientific choices live in the hashed reproducibility.json; they are fixed,
# including seed 42, analysis 256, dynamic shortening and batching configuration.

## Verify uploaded source and set up dependencies
Follow the same source ZIP, protected dependency and pinned model-cache conventions as notebook 03. Upload `spw_kaggle_source.zip` as a Dataset or attach its extracted contents, set `UPLOAD_SOURCE`, enable a Tesla T4 and Internet for streaming. The source hash is checked before imports; stale source directories fail explicitly. No existing attack is run.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
SOURCE_MANIFEST_SHA256 = '8d0200e7076d992441a0e920b81fd83c46f554ba015323623d28ad30a852091a'
if not (PROJECT_ROOT / "spw_code_attack/source_manifest.json").is_file():
    if PROJECT_ROOT.exists():
        raise FileExistsError("Incomplete PROJECT_ROOT; select a fresh directory")
    if (UPLOAD_SOURCE / "spw_code_attack/source_manifest.json").is_file():
        source_manifest_path = UPLOAD_SOURCE / "spw_code_attack/source_manifest.json"
        if hashlib.sha256(source_manifest_path.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
            raise ValueError("Outdated SPW-inspired source package")
        source_manifest = json.loads(source_manifest_path.read_text())
        PROJECT_ROOT.mkdir(parents=True)
        for name in list(source_manifest) + ["spw_code_attack/source_manifest.json"]:
            destination = (PROJECT_ROOT / name).resolve()
            if not destination.is_relative_to(PROJECT_ROOT.resolve()):
                raise ValueError("Unsafe source path")
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(UPLOAD_SOURCE / name, destination)
    else:
        with zipfile.ZipFile(UPLOAD_ZIP) as archive:
            for entry in archive.infolist():
                if not (PROJECT_ROOT / entry.filename).resolve().is_relative_to(PROJECT_ROOT.resolve()):
                    raise ValueError("Unsafe archive path")
            zipped_manifest = archive.read('spw_code_attack/source_manifest.json')
            if hashlib.sha256(zipped_manifest).hexdigest() != SOURCE_MANIFEST_SHA256:
                raise ValueError('Outdated SPW-inspired source ZIP')
            allowed = set(json.loads(zipped_manifest)) | {'spw_code_attack/source_manifest.json'}
            if set(archive.namelist()) != allowed:
                raise ValueError('Unexpected archive files')
            archive.extractall(PROJECT_ROOT)
manifest_path = PROJECT_ROOT / "spw_code_attack/source_manifest.json"
if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
    raise ValueError("Source manifest mismatch; use a fresh PROJECT_ROOT and rebuilt package")
source_manifest = json.loads(manifest_path.read_text())
for name, expected in source_manifest.items():
    if hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest() != expected:
        raise ValueError("Source mismatch: " + name)
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "SERSEM-Poisoned-Chalice-Competition-2026"
sys.path.insert(0, str(UPSTREAM / "code"))

In [ ]:
# Install the existing pinned baseline dependencies; preserve Kaggle's numerical stack.
from shared.kaggle_setup import install_dependencies, validate_environment
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
packages = ["libenchant-2-2", "hunspell-en-us"]
status = subprocess.run(["dpkg-query", "-W", "-f=${Status}\n", *packages], capture_output=True, text=True)
if status.returncode != 0 or status.stdout.splitlines().count("install ok installed") != len(packages):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *packages], check=True)
install_dependencies(PROJECT_ROOT / "shared/requirements-kaggle.txt", WORK_ROOT)
environment = validate_environment(PROJECT_ROOT / "shared/requirements-kaggle.txt")

In [ ]:
# CPU synthetic checks only, including a tiny randomly initialized model.
# No pretrained weights or dataset access; PyTorch is provided by Kaggle setup.
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_spw.py','-v'],
               cwd=PROJECT_ROOT,check=True)

## Validate all saved identities before selecting rows
Complete 1600/200/200/1000 manifests are required before smoke selection. No new splits. Smoke is stratified 80/20/20/30 using **seed 42**. This can differ from notebook 03's smoke cohort: attach full adaptive held-out scores, or a score file containing every exact selected identity. Missing comparison IDs fail before model inference. Source versions and numerical/API environment match the baseline exactly.

In [ ]:
import copy, random
import numpy as np
import pandas as pd
from adaptive_token_attack.data import resolve_paths,load_run,select_partitions,frame_fingerprint
from adaptive_token_attack.features import load_frozen_target
from spw_code_attack.cache import Cache,fingerprint
from spw_code_attack.data import scoring_identity,prepare_plans,fixed_configuration
from spw_code_attack.inference import TorchPredictor
from spw_code_attack.runner import run_scores
from spw_code_attack.evaluation import exact_match,evaluate
from spw_code_attack import METHOD_NAME,METHOD_STATUS
from shared.resource_pipeline import Monitor,restore_run,file_hash
from shared.resource_data import SourceStore
from shared.sersem_baseline import write_json
fixed = fixed_configuration()
print(METHOD_NAME, METHOD_STATUS)
paths = resolve_paths(BASELINE_RUN,PATH_OVERRIDES,INPUT_ROOT)
original,saved_scores,base_cfg = load_run(paths)
if SMOKE_SIZES != fixed['smoke_counts']:
    raise ValueError('Smoke counts are fixed to 80/20/20/30')
partitions,selection = select_partitions(original,MODE,SMOKE_SIZES,seed=fixed['seed'])
if set(pd.concat(list(original.values())).language) != {'Go','Java','Python','Ruby','Rust'}:
    raise ValueError('Unexpected saved programming languages')
if base_cfg['versions'] != environment['versions']:
    raise ValueError('Restore the saved baseline numerical/API environment before inference/comparison')
if ADAPTIVE_SCORES:
    adaptive_path = Path(ADAPTIVE_SCORES)
else:
    candidates = sorted(INPUT_ROOT.rglob('heldout_adaptive_scores.parquet'))
    if len(candidates) != 1:
        raise ValueError('Set ADAPTIVE_SCORES explicitly: '+str(candidates))
    adaptive_path = candidates[0]
adaptive_scores = pd.read_parquet(adaptive_path)
exact_match(partitions['heldout'],adaptive_scores,['adaptive_score'])
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {'.','..'}:
    raise ValueError('RUN_ID must be a directory name')
if OUT.resolve().is_relative_to(INPUT_ROOT.resolve()):
    raise ValueError('OUT must be writable and separate from inputs')
OUT.mkdir(parents=True,exist_ok=True)
if RESUME_FROM:
    restore_run(RESUME_FROM,OUT)
source_cfg = copy.deepcopy(base_cfg)
source_cfg['resources'].update(data_access=SOURCE_ACCESS,text_cache_gib=.1,
    session_budget_minutes=SESSION_BUDGET_MINUTES,disk_reserve_gib=DISK_RESERVE_GIB)
monitor = Monitor(source_cfg,OUT)
monitor.guard()
print('Selected:',{k:len(v) for k,v in partitions.items()})

## Pin context, tokenizer and scientific configuration
Frozen FP16 model, FP32 log-softmax, FP64 probability/divergence reductions. Baseline tokenization uses `add_special_tokens=True`, right truncation, attention/special masks and character offsets. Eligible targets are predictable non-special tokens with nonempty offsets; select the first 256. Excluded targets remain in preceding context. Position zero is never scored, no BOS is invented, and shortened suffixes reset positions to zero. Short samples are retained and flagged; empty target spans fail. No external prefix pool is used.

Cache preparation uses metadata/tokenizers only. No saved baseline loss or probabilities are presumed compatible. Every scientific choice and source file is hashed before scoring. The full published method is unavailable; `paper_faithful=False` is mandatory.

In [ ]:
from shared.cache_setup import ensure_model_snapshot
from transformers import AutoConfig,AutoTokenizer
cache_dir = MODEL_CACHE_DIR or base_cfg['resources']['model_cache_dir']
ensure_model_snapshot(base_cfg['model_id'],base_cfg['model_revision'],cache_dir,
    ALLOW_MODEL_DOWNLOAD,DISK_RESERVE_GIB,WORK_ROOT,OUT/'model_cache_setup.json')
options = dict(revision=base_cfg['model_revision'],cache_dir=str(cache_dir),local_files_only=MODEL_LOCAL_FILES_ONLY)
tokenizer = AutoTokenizer.from_pretrained(base_cfg['model_id'],use_fast=True,**options)
model_config = AutoConfig.from_pretrained(base_cfg['model_id'],**options)
identity = scoring_identity(base_cfg,tokenizer,model_config,source_manifest)
experiment = dict(schema='spw-inspired-experiment-v1',method_name=METHOD_NAME,method_status=METHOD_STATUS,
    paper_faithful=False,mode=MODE,selection=selection,scoring=identity,
    source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    inputs={k:hashlib.sha256(p.read_bytes()).hexdigest() for k,p in paths.items()},
    adaptive_scores_sha256=hashlib.sha256(adaptive_path.read_bytes()).hexdigest())
if (OUT/'config.json').exists() and json.loads((OUT/'config.json').read_text()) != experiment:
    raise ValueError('Execution identity changed; choose a new RUN_ID')
write_json(OUT/'config.json',experiment)
write_json(OUT/'reproducibility.json',fixed)
write_json(OUT/'research_access.json',json.loads((PROJECT_ROOT/'spw_code_attack/research_access.json').read_text()))
write_json(OUT/'source_version_hashes.json',dict(source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    sources=source_manifest,versions=environment,scoring_sha256=fingerprint(identity)))
write_json(OUT/'input_paths.json',dict(**{k:str(v) for k,v in paths.items()},adaptive_scores=str(adaptive_path)))
for role,frame in partitions.items():
    frame.to_parquet(OUT/(role+'_manifest.parquet'),index=False)
write_json(OUT/'run_manifest.json',dict(status='preparing',method_name=METHOD_NAME,paper_faithful=False,
    mode=MODE,selection=selection,scoring_sha256=fingerprint(identity)))
cache = Cache(OUT/'spw_cache.sqlite')
if CACHE_REUSE_FROM:
    monitor.guard(Path(CACHE_REUSE_FROM).stat().st_size)
    cache.reuse_from(CACHE_REUSE_FROM)
source_store = SourceStore(source_cfg,OUT,monitor)
plans = prepare_plans(partitions,tokenizer,identity,cache,source_store,monitor)
print('Prepared plans:',len(plans))

## Dynamic shortening with cached probabilities and terminal states
A teacher-forced pass collects original-prefix true-token and greedy-token probabilities. For each target independently, halve the accepted suffix length and evaluate it. Continue if its greedy token matches the original-prefix greedy token; stop at first mismatch or length one. This is **our approximation rule**, not a verified published rule.

Group requests by exact prefix, then equal length, in batches of four without padding. No approximate KV sharing across shortened prefixes. Cache compact FP16 terminal states losslessly as well as probabilities: adding a different true token for a known prefix requires only an LM-head projection. The model's native decoder/head are used under `torch.inference_mode()` and all parameters are frozen. The canonical head projection uses one row in both initial queries and cache extensions. Native original-prefix KV chunks are 256 tokens.

Every completed prediction commits before the control flow advances. A partial run replays decisions from cached values. Valid complete samples skip all model calls; complete plan caches skip source iteration. Resource checks stop safely between operations. The cache can reach several GiB; disk reserve checks never delete cached work. Save outputs and resume if the session cannot finish.

In [ ]:
def predictor_factory():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Missing predictions require Kaggle GPU')
    monitor.guard()
    random.seed(42);np.random.seed(42);torch.manual_seed(42);torch.cuda.manual_seed_all(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    model,loaded_tokenizer = load_frozen_target(base_cfg,cache_dir,MODEL_LOCAL_FILES_ONLY)
    if hashlib.sha256(loaded_tokenizer.backend_tokenizer.to_str().encode()).hexdigest() != identity['tokenizer_sha256']:
        raise ValueError('Inference tokenizer differs from prepared plans')
    return TorchPredictor(model)
write_json(OUT/'run_manifest.json',dict(status='scoring',method_name=METHOD_NAME,method_status=METHOD_STATUS,
    paper_faithful=False,mode=MODE,selection=selection,scoring_sha256=fingerprint(identity),
    source_manifest_sha256=SOURCE_MANIFEST_SHA256))
scored,timing = run_scores(partitions,plans,identity,cache,predictor_factory,monitor,OUT)
import gc,torch
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
print(json.dumps(timing,indent=2))

## Calibration and exact paired evaluation
Select one global threshold only from the separate calibration partition at empirical FPR <=1%. Higher `spw_inspired_score` is member-like by the frozen negative-divergence definition; no sign search. Report deployed calibration-threshold TPR/FPR separately from held-out diagnostic ROC thresholds. Compare SPW-inspired scores with static SERSEM, adaptive SERSEM, LUMIA and saved fusion on exactly matched identities/labels/content.

Full coverage is 1000/1000. Smoke covers 30/1000 original held-out identities and is preliminary; low-FPR estimates are coarse. Pairing matches identities: baseline methods keep their original spans, while this experiment analyzes 256 eligible targets. Timing reports transformer and LM-head forwards separately, cache hits, session time and durable call accounting. Hard-killed unfinished operations are flagged as unknown instead of inventing exact historical counts.

In [ ]:
report = evaluate(scored,partitions,saved_scores,adaptive_scores,OUT,MODE,timing=timing)
monitor.guard()
cache.close()
write_json(OUT/'run_manifest.json',dict(status='complete',method_name=METHOD_NAME,method_status=METHOD_STATUS,
    paper_faithful=False,mode=MODE,selection=selection,scoring_sha256=fingerprint(identity),
    source_manifest_sha256=SOURCE_MANIFEST_SHA256,input_hashes=experiment['inputs'],
    adaptive_scores_sha256=experiment['adaptive_scores_sha256'],
    selected_counts={k:len(v) for k,v in partitions.items()},coverage=report['coverage'],timing=timing,
    span_summary=dict(short_samples=int(scored.short_sample.sum()),
        context_truncated_samples=int((scored.context_or_baseline_truncated_tokens>0).sum())),
    artifacts={p.name:file_hash(p) for p in OUT.iterdir()
        if p.is_file() and p.name in {'config.json','metrics.json','spw_scores.parquet','matched_comparison.parquet',
            'spw_cache.sqlite','source_version_hashes.json','threshold.json','timing.json','reproducibility.json','research_access.json'}}))
print(json.dumps(report,indent=2))

## Persist, resume, and interpret
Save the **entire OUT directory** via Kaggle Save Version **with outputs**. It contains configuration, source/version hashes, `metrics.json`, `spw_scores.parquet`, `matched_comparison.parquet`, `spw_cache.sqlite`, `run_manifest.json`, fixed reproducibility specification, research-access record, selected manifests, calibrated threshold, timing, progress and resource/call logs. Per-token original/shortened probabilities, greedy IDs, accepted lengths, weights and divergence terms remain in the database for audit.

Same run: attach saved OUT, keep MODE/RUN_ID/scientific identity, set `RESUME_FROM` to that directory. Smoke → full: choose `MODE='full'`, a new full RUN_ID, `RESUME_FROM=''`, and optionally `CACHE_REUSE_FROM` pointing to the saved smoke database; attach full saved adaptive scores. Cache imports retain compatible probabilities/plans/results but do not import old call logs. Full after smoke never imports a fitted threshold.

Source/config changes invalidate keys. Only operational paths and resource budgets may change on resume. Stop running cells before copying SQLite files; preserve any rollback journal left by interruption. No automatic dataset/model/cache deletion. A call killed before its transaction commits must be repeated; completed cache entries prevent repeat model queries. Original probabilities and shortened-prefix control flow are cached independently of labels.

This is an SPW-inspired approximation whose equations are stated in the README. The published dynamic shortening criterion, weight formula, divergence/combination and defaults remain unverified. These results must not be presented as full original SPW-MIA or as equivalent to the paper.